In [ ]:
from google.colab import userdata
from pathlib import Path
import os, subprocess

repo = Path('/content/Dissertation')
url = 'https://github.com/520/Dissertation.git'
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('请在 Colab Secrets 中添加 GITHUB_TOKEN')

askpass = Path('/tmp/github_askpass.sh')
askpass.write_text("#!/bin/sh\ncase \"$1\" in\n*Username*) echo x-access-token;;\n*Password*) echo \"$GITHUB_TOKEN\";;\nesac\n")
askpass.chmod(0o700)
env = {**os.environ, 'GITHUB_TOKEN': token, 'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0'}

def git(*args):
    result = subprocess.run(['git', *map(str, args)], env=env, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if (repo / '.git').is_dir():
    print('仓库已存在，正在与 GitHub main 同步……')
    git('-C', repo, 'remote', 'set-url', 'origin', url)
    git('-C', repo, 'fetch', '--prune', 'origin', 'main')
    git('-C', repo, 'reset', '--hard', 'origin/main')
elif repo.exists():
    raise RuntimeError(f'{repo} 已存在但不是 Git 仓库，请删除或改名后重试')
else:
    print('仓库不存在，正在克隆……')
    git('clone', '--branch', 'main', '--single-branch', url, repo)

os.chdir(repo)
print('同步完成，当前 commit：', git('-C', repo, 'rev-parse', '--short', 'HEAD'))

In [ ]:
!pip install -q ultralytics torchao 'nvidia-modelopt[torch,onnx]'

In [ ]:
%cd /content/Dissertation
!python -m QAT.torch_qat